<a href="https://colab.research.google.com/github/Aswin-k61/NLP_Repo/blob/main/Neural_network_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import string

from google.colab import files
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

nltk.download('stopwords')



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
df=pd.read_csv('/content/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [ ]:
df.shape

(50000, 2)

In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df=df.sample(
    5000,
    random_state=42
)

In [ ]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
}
)

In [ ]:
from nltk.corpus.reader import WordNetCorpusReader
stop_words=set(stopwords.words('english'))
stemmer=PorterStemmer()

def preprocess(text):
  text=text.lower()
  text=re.sub('<.*?>','',text)

 # remove punctuation
  text=text.translate(
  str.maketrans('','',string.punctuation))

 # tokenization
  words=text.split()

 # stopword removal
  words=[
    word for word in words
    if word not in stop_words
  ]

  #stemming
  words=[
      stemmer.stem(word)
      for word in words
   ]


  return " ".join(words)


In [ ]:
df['clean_review']=df['review'].apply(preprocess)

In [ ]:
vectorizer=TfidfVectorizer(
    max_features=5000,
)
X=vectorizer.fit_transform(df['clean_review']).toarray()
y=df['sentiment']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
model=Sequential()
model.add(Dense(128,activation='relu',input_shape=(X_train.shape[1],)))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
)

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7952 - loss: 0.4826
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9477 - loss: 0.1540
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9918 - loss: 0.0456
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9992 - loss: 0.0102
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 0.0029
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 0.0015
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 9.5433e-04
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 1.0000 - loss: 6.6837e-04
Epoch 9/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 1.0000 - loss: 4.9406e-04
Epoch 10/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 3.7815e-04
Epoch 11/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 2.9699e-04
Epoch 12/15
125/125 ━━━━━━━━

In [ ]:
loss,accuracy=model.evaluate(X_test,y_test)
print("Accuracy:",accuracy)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8250 - loss: 0.8743
Accuracy: 0.824999988079071


In [ ]:
review=["This movie was fantastic"]
clean=preprocess(review[0])
vector=vectorizer.transform(
    [clean]
).toarray()
pred=model.predict(vector)
if pred>0.5:
  print("Positive")
else:
  print("Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Positive
